# Detail Walkthrough — Web Scraping & API Integration

Complete compilation of all steps from the `steps/` folder, notebook edition.

| Step | Topic |
|------|-------|
| 1 | Pipeline Overview |
| 2 | BeautifulSoup — Scrape 1 Item |
| 3 | BeautifulSoup — Scrape 1 Page |
| 4 | BeautifulSoup — Multiple Pages (Pagination) |
| 5 | Selenium — Dynamic Website |
| 6 | Selenium — XPath |
| 7 | API (dummyjson) |
| 8 | Data Cleaning with pandas |
| 9a | Save to SQLite |
| 9b | Save to PostgreSQL (NeonDB) |

---
## Step 1 — Pipeline Overview

A **data pipeline** is an automated flow for moving data from a **source** to a **destination**, through a process that is **structured** and **repeatable**.

```mermaid
flowchart TD
    A[Scrape 1 Item] --> B[Scrape 1 Page]
    B --> C[Build a Dictionary List]
    C --> D[Scrape Several Pages]
    D --> E[Clean the Data]
    E --> F[Open a Database Connection]
    F --> G[Create Table & Insert Data]
```

**Data sources used in this project:**
- Static  (BeautifulSoup) : https://books.toscrape.com
- Dynamic (Selenium)      : https://quotes.toscrape.com/js
- API                     : https://dummyjson.com/products

In [1]:
PIPELINE_STAGES = [
    "Scrape 1 item",
    "Scrape 1 page",
    "Build a Dictionary List",
    "Scrape Several Pages",
    "Clean the Data",
    "Open a Database Connection",
    "Create the Table and Insert Data",
]

print("=" * 50)
print("PROJECT FLOW: Web Scraping & API Integration")
print("=" * 50)
for i, stage in enumerate(PIPELINE_STAGES, start=1):
    print(f"  [{i}] {stage}")
    if i < len(PIPELINE_STAGES):
        print("        |")
        print("        v")
print("=" * 50)

PROJECT FLOW: Web Scraping & API Integration
  [1] Scrape 1 item
        |
        v
  [2] Scrape 1 page
        |
        v
  [3] Build a Dictionary List
        |
        v
  [4] Scrape Several Pages
        |
        v
  [5] Clean the Data
        |
        v
  [6] Open a Database Connection
        |
        v
  [7] Create the Table and Insert Data


---
## Step 2 — BeautifulSoup: Scrape 1 Item

**BeautifulSoup** is a library for parsing HTML so we can extract data based on tags and attributes.

Two main methods:
- `find()` → grab the **first** matching element
- `find_all()` → grab **all** matching elements

Target: `https://books.toscrape.com/` — extract the very first book only.

In [2]:
import requests
from bs4 import BeautifulSoup

URL = "https://books.toscrape.com/"

# 1) Fetch the page
response = requests.get(URL, timeout=10)
response.raise_for_status()
response.encoding = "utf-8"  # ensures the £ symbol displays correctly

# 2) Parse the HTML
soup = BeautifulSoup(response.text, "html.parser")

# 3) Grab the FIRST item with find()
#    Each book lives inside <article class="product_pod">
item = soup.find("article", class_="product_pod")

# 4) Extract individual fields
title  = item.find("h3").find("a")["title"]
price  = item.find("p", class_="price_color").text
rating = item.find("p", class_="star-rating")["class"][1]  # e.g. "Three"
stock  = item.find("p", class_="instock availability").text.strip()

print("=== Scrape 1 Item ===")
print("Title  :", title)
print("Price  :", price)
print("Rating :", rating)
print("Stock  :", stock)

=== Scrape 1 Item ===
Title  : A Light in the Attic
Price  : £51.77
Rating : Three
Stock  : In stock


---
## Step 3 — BeautifulSoup: Scrape 1 Page (List)

Use `find_all()` to grab **all** products on a single page, then store them as a **list of dicts**.

In [3]:
import requests
from bs4 import BeautifulSoup

URL = "https://books.toscrape.com/"

def scrape_page(url: str) -> list[dict]:
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    response.encoding = "utf-8"
    soup = BeautifulSoup(response.text, "html.parser")

    products: list[dict] = []
    for item in soup.find_all("article", class_="product_pod"):  # all books on this page
        products.append({
            "name"  : item.find("h3").find("a")["title"],
            "price" : item.find("p", class_="price_color").text,
            "rating": item.find("p", class_="star-rating")["class"][1],
            "link"  : item.find("h3").find("a")["href"],
        })
    return products

data = scrape_page(URL)
print(f"=== Successfully scraped {len(data)} products from 1 page ===")
for p in data[:5]:
    print(p)
print("...")

=== Successfully scraped 20 products from 1 page ===
{'name': 'A Light in the Attic', 'price': '£51.77', 'rating': 'Three', 'link': 'catalogue/a-light-in-the-attic_1000/index.html'}
{'name': 'Tipping the Velvet', 'price': '£53.74', 'rating': 'One', 'link': 'catalogue/tipping-the-velvet_999/index.html'}
{'name': 'Soumission', 'price': '£50.10', 'rating': 'One', 'link': 'catalogue/soumission_998/index.html'}
{'name': 'Sharp Objects', 'price': '£47.82', 'rating': 'Four', 'link': 'catalogue/sharp-objects_997/index.html'}
{'name': 'Sapiens: A Brief History of Humankind', 'price': '£54.23', 'rating': 'Five', 'link': 'catalogue/sapiens-a-brief-history-of-humankind_996/index.html'}
...


---
## Step 4 — BeautifulSoup: Multiple Pages (Pagination)

Loop through several pages and collect all products into one big list.

URL pattern: `https://books.toscrape.com/catalogue/page-{N}.html`

In [4]:
import requests
from bs4 import BeautifulSoup

BASE_URL = "https://books.toscrape.com/catalogue/page-{}.html"

def scrape_page(url: str) -> list[dict]:
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    response.encoding = "utf-8"
    soup = BeautifulSoup(response.text, "html.parser")
    products: list[dict] = []
    for item in soup.find_all("article", class_="product_pod"):
        products.append({
            "name"  : item.find("h3").find("a")["title"],
            "price" : item.find("p", class_="price_color").text,
            "rating": item.find("p", class_="star-rating")["class"][1],
            "link"  : item.find("h3").find("a")["href"],
        })
    return products

def scrape_multiple_pages(page_count: int = 3) -> list[dict]:
    all_products: list[dict] = []
    for page in range(1, page_count + 1):
        url = BASE_URL.format(page)
        print(f"-> Scraping page {page}: {url}")
        all_products.extend(scrape_page(url))
    return all_products

data = scrape_multiple_pages(page_count=3)
print(f"\n=== Total {len(data)} products from 3 pages ===")
print("First record :", data[0])
print("Last record  :", data[-1])

-> Scraping page 1: https://books.toscrape.com/catalogue/page-1.html


-> Scraping page 2: https://books.toscrape.com/catalogue/page-2.html


-> Scraping page 3: https://books.toscrape.com/catalogue/page-3.html



=== Total 60 products from 3 pages ===
First record : {'name': 'A Light in the Attic', 'price': '£51.77', 'rating': 'Three', 'link': 'a-light-in-the-attic_1000/index.html'}
Last record  : {'name': 'The Natural History of Us (The Fine Art of Pretending #2)', 'price': '£45.22', 'rating': 'Three', 'link': 'the-natural-history-of-us-the-fine-art-of-pretending-2_941/index.html'}


---
## Step 5 — Selenium: Dynamic Website

**Why Selenium?**  
Fetching `https://quotes.toscrape.com/js/` with `requests` returns **empty content** — the quotes are rendered by JavaScript inside the browser. Selenium drives a real browser, so the JavaScript actually runs.

### ❌ Bad example — `time.sleep` (avoid this)
```python
import time
driver.get(url)
time.sleep(3)  # fixed delay — may be too short or wasteful
quotes = driver.find_elements(By.CLASS_NAME, "quote")  # could still be 0!
```

### ✅ Good example — `WebDriverWait` + `expected_conditions`
Polls the DOM every 500 ms and proceeds **as soon as** the element appears. If it never appears → raises a clear `TimeoutException`.

In [5]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

URL = "https://quotes.toscrape.com/js/"

def build_driver(headless: bool = True) -> webdriver.Chrome:
    options = Options()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--window-size=1920,1080")
    return webdriver.Chrome(options=options)

driver = build_driver()
try:
    driver.get(URL)

    # Wait until at least 1 .quote element appears in the DOM (max 10 s)
    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.CLASS_NAME, "quote"))
    )

    quotes = driver.find_elements(By.CLASS_NAME, "quote")
    print(f"=== Got {len(quotes)} quotes from the dynamic website ===")

    for q in quotes[:5]:
        text   = q.find_element(By.CLASS_NAME, "text").text
        # CSS selector: target specifically the <small> tag with class "author"
        author = q.find_element(By.CSS_SELECTOR, "small.author").text
        print(f"- {text}  -- {author}")
finally:
    driver.quit()  # always close the browser

=== Got 10 quotes from the dynamic website ===
- “The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”  -- Albert Einstein
- “It is our choices, Harry, that show what we truly are, far more than our abilities.”  -- J.K. Rowling
- “There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.”  -- Albert Einstein
- “The person, be it gentleman or lady, who has not pleasure in a good novel, must be intolerably stupid.”  -- Jane Austen
- “Imperfection is beauty, madness is genius and it's better to be absolutely ridiculous than absolutely boring.”  -- Marilyn Monroe


---
## Step 6 — Selenium: XPath

**XPath** is a query language for navigating and selecting nodes inside the DOM (HTML/XML).

| XPath | Meaning |
|-------|---------|
| `//h1` | all `<h1>` tags |
| `//p[@class="plot"]` | `<p>` whose class is **exactly** `"plot"` |
| `//p[contains(@class,"plot")]` | `<p>` whose class **contains** `"plot"` |
| `//*[@id="x"]` | any element with `id="x"` |
| `//p[1]` | the first `<p>` |
| `.//span` | `<span>` relative to the current element |

In [6]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

URL = "https://quotes.toscrape.com/js/"

def build_driver(headless: bool = True) -> webdriver.Chrome:
    options = Options()
    if headless:
        options.add_argument("--headless=new")
    options.add_argument("--window-size=1920,1080")
    return webdriver.Chrome(options=options)

driver = build_driver()
try:
    driver.get(URL)

    # Wait until the first quote div appears
    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.XPATH, '//div[@class="quote"]'))
    )

    # XPath: select all <div class="quote">
    quotes = driver.find_elements(By.XPATH, '//div[@class="quote"]')
    print(f"=== {len(quotes)} quotes via XPath ===")

    for q in quotes[:5]:
        # Relative XPath (starts with ".") → searched WITHIN this quote element
        text   = q.find_element(By.XPATH, './/span[@class="text"]').text
        author = q.find_element(By.XPATH, './/small[@class="author"]').text
        print(f"- {text}  -- {author}")

    # Another example: grab the page <h1> directly
    title = driver.find_element(By.XPATH, "//h1").text
    print("\nPage title (//h1):", title)
finally:
    driver.quit()

=== 10 quotes via XPath ===
- “The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”  -- Albert Einstein
- “It is our choices, Harry, that show what we truly are, far more than our abilities.”  -- J.K. Rowling
- “There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.”  -- Albert Einstein
- “The person, be it gentleman or lady, who has not pleasure in a good novel, must be intolerably stupid.”  -- Jane Austen
- “Imperfection is beauty, madness is genius and it's better to be absolutely ridiculous than absolutely boring.”  -- Marilyn Monroe

Page title (//h1): Quotes to Scrape


---
## Step 7 — API (dummyjson)

An **API** (Application Programming Interface) is the *official* way to fetch structured data directly from a provider's system.

API characteristics:
- Data is usually returned as **JSON / XML**
- Stable & structured
- Faster and more reliable than scraping
- Often has rate limits & authentication

Target: `https://dummyjson.com/products`

In [7]:
import requests

URL = "https://dummyjson.com/products"

def fetch_api_data(limit: int = 10) -> list[dict]:
    # Many APIs accept query parameters, e.g. limit & select
    params = {"limit": limit, "select": "title,price,category,brand"}
    response = requests.get(URL, params=params, timeout=10)
    response.raise_for_status()

    data = response.json()
    # dummyjson structure: {"products": [...], "total": ..., ...}
    return data["products"]

products = fetch_api_data(limit=10)
print(f"=== Successfully fetched {len(products)} products from the API ===")
for p in products[:5]:
    print(p)

=== Successfully fetched 10 products from the API ===
{'id': 1, 'title': 'Essence Mascara Lash Princess', 'price': 9.99, 'category': 'beauty', 'brand': 'Essence'}
{'id': 2, 'title': 'Eyeshadow Palette with Mirror', 'price': 19.99, 'category': 'beauty', 'brand': 'Glamour Beauty'}
{'id': 3, 'title': 'Powder Canister', 'price': 14.99, 'category': 'beauty', 'brand': 'Velvet Touch'}
{'id': 4, 'title': 'Red Lipstick', 'price': 12.99, 'category': 'beauty', 'brand': 'Chic Cosmetics'}
{'id': 5, 'title': 'Red Nail Polish', 'price': 8.99, 'category': 'beauty', 'brand': 'Nail Couture'}


---
## Step 8 — Data Cleaning with pandas

Before saving to the database, the raw data needs to be cleaned:
1. **Reformat** the price column: `"£51.77"` → `51.77` (float)
2. **Convert** text ratings (`"Three"`) → numbers (`3`)
3. **Drop** empty rows and duplicates

In [8]:
import re

import pandas as pd
import requests
from bs4 import BeautifulSoup

BASE_URL   = "https://books.toscrape.com/catalogue/page-{}.html"
RATING_MAP = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

def scrape_multiple_pages(page_count: int = 2) -> list[dict]:
    all_products: list[dict] = []
    for page in range(1, page_count + 1):
        resp = requests.get(BASE_URL.format(page), timeout=10)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")
        for item in soup.find_all("article", class_="product_pod"):
            all_products.append({
                "name"  : item.find("h3").find("a")["title"],
                "price" : item.find("p", class_="price_color").text,
                "rating": item.find("p", class_="star-rating")["class"][1],
            })
    return all_products

def clean(data: list[dict]) -> pd.DataFrame:
    df = pd.DataFrame(data)

    # Clean price: "£51.77" -> 51.77 (float)
    df["price"] = (
        df["price"]
        .apply(lambda x: re.sub(r"[^0-9.]", "", x))
        .astype(float)
    )

    # Convert text rating to integer
    df["rating"] = df["rating"].map(RATING_MAP)

    # Drop empty rows and duplicates
    df = df.dropna().drop_duplicates().reset_index(drop=True)
    return df

raw_data = scrape_multiple_pages(page_count=2)
print(f"Raw data: {len(raw_data)} rows")
print("Sample raw:", raw_data[:2])

df_clean = clean(raw_data)
print("\n=== Data after cleaning ===")
print(df_clean.head())
print("\nColumn types:")
print(df_clean.dtypes)

Raw data: 40 rows
Sample raw: [{'name': 'A Light in the Attic', 'price': 'Â£51.77', 'rating': 'Three'}, {'name': 'Tipping the Velvet', 'price': 'Â£53.74', 'rating': 'One'}]

=== Data after cleaning ===
                                    name  price  rating
0                   A Light in the Attic  51.77       3
1                     Tipping the Velvet  53.74       1
2                             Soumission  50.10       1
3                          Sharp Objects  47.82       4
4  Sapiens: A Brief History of Humankind  54.23       5

Column types:
name          str
price     float64
rating      int64
dtype: object


---
## Step 9a — Save to SQLite

**SQLite** is built into Python — no server installation needed. Great for local development and practice.

```mermaid
flowchart LR
    A[Scrape & Clean] --> B[sqlite3.connect]
    B --> C[CREATE TABLE]
    C --> D[executemany INSERT]
    D --> E[commit]
    E --> F[Verify: SELECT]
```

In [9]:
import re
import sqlite3
from pathlib import Path

import requests
from bs4 import BeautifulSoup

BASE_URL   = "https://books.toscrape.com/catalogue/page-{}.html"
RATING_MAP = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}
DB_PATH    = Path("../data/ecommerce.db")

def scrape_and_clean(page_count: int = 2) -> list[dict]:
    results: list[dict] = []
    for page in range(1, page_count + 1):
        resp = requests.get(BASE_URL.format(page), timeout=10)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")
        for item in soup.find_all("article", class_="product_pod"):
            price_text = item.find("p", class_="price_color").text
            results.append({
                "name"  : item.find("h3").find("a")["title"],
                "price" : float(re.sub(r"[^0-9.]", "", price_text)),
                "rating": RATING_MAP.get(item.find("p", class_="star-rating")["class"][1]),
            })
    return results

def save_to_sqlite(data: list[dict], db_path: Path = DB_PATH) -> None:
    db_path.parent.mkdir(parents=True, exist_ok=True)

    # 1) Open connection (creates the file if it doesn't exist)
    conn = sqlite3.connect(db_path)
    cur  = conn.cursor()

    # 2) Create table
    cur.execute("""
        CREATE TABLE IF NOT EXISTS products (
            id     INTEGER PRIMARY KEY AUTOINCREMENT,
            name   TEXT    NOT NULL,
            price  REAL,
            rating INTEGER
        )
    """)

    # 3) Insert all rows at once
    cur.executemany(
        "INSERT INTO products (name, price, rating) VALUES (:name, :price, :rating)",
        data,
    )

    conn.commit()
    print(f"[SQLite] {len(data)} rows saved to {db_path}")

    # 4) Verify: read back the first 5 rows
    print("\nSample rows from database:")
    for row in cur.execute("SELECT * FROM products LIMIT 5"):
        print(row)

    conn.close()

data = scrape_and_clean(page_count=2)
print(f"Ready to insert {len(data)} rows.\n")
save_to_sqlite(data)

Ready to insert 40 rows.

[SQLite] 40 rows saved to ../data/ecommerce.db

Sample rows from database:
(1, 'A Light in the Attic', 51.77, 3)
(2, 'Tipping the Velvet', 53.74, 1)
(3, 'Soumission', 50.1, 1)
(4, 'Sharp Objects', 47.82, 4)
(5, 'Sapiens: A Brief History of Humankind', 54.23, 5)


---
## Step 9b — Save to PostgreSQL (NeonDB)

**PostgreSQL** is a production-grade database server. Here we use **NeonDB** — a managed, serverless PostgreSQL in the cloud.

```mermaid
flowchart LR
    A[Scrape & Clean] --> B[psycopg2.connect]
    B --> C[CREATE TABLE]
    C --> D[execute_values INSERT]
    D --> E[commit]
    E --> F[Verify: SELECT]
    B -- SSL required --> G[(NeonDB\nCloud)]
```

**Requirement:**
```bash
uv add psycopg2-binary
```

Set credentials via environment variables (recommended) or directly in the config cell below:
```bash
export PGHOST='ep-silent-bar-adim0vxo-pooler.c-2.us-east-1.aws.neon.tech'
export PGDATABASE='neondb'
export PGUSER='neondb_owner'
export PGPASSWORD='your_password'
export PGSSLMODE='require'
export PGCHANNELBINDING='require'
```

In [10]:
import os

# ─── NeonDB Configuration ─────────────────────────────────────────────────────
# Replace the fallback values below with your actual NeonDB credentials,
# or set them as environment variables before running the notebook.

PG_CONFIG = {
    "host"           : os.getenv("PGHOST",           "ep-silent-bar-adim0vxo-pooler.c-2.us-east-1.aws.neon.tech"),
    "dbname"         : os.getenv("PGDATABASE",       "neondb"),
    "user"           : os.getenv("PGUSER",           "sample_user"),
    "password"       : os.getenv("PGPASSWORD",       "sample_password"),
    "sslmode"        : os.getenv("PGSSLMODE",        "require"),
    "channel_binding": os.getenv("PGCHANNELBINDING", "require"),
}

print("Config in use:")
for k, v in PG_CONFIG.items():
    display = "*" * len(v) if k == "password" else v
    print(f"  {k:20s}: {display}")

Config in use:
  host                : ep-silent-bar-adim0vxo-pooler.c-2.us-east-1.aws.neon.tech
  dbname              : neondb
  user                : sample_user
  password            : ***************
  sslmode             : require
  channel_binding     : require


In [11]:
import re

import psycopg2
import psycopg2.extras
import requests
from bs4 import BeautifulSoup

BASE_URL   = "https://books.toscrape.com/catalogue/page-{}.html"
RATING_MAP = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}

def scrape_and_clean(page_count: int = 2) -> list[dict]:
    results: list[dict] = []
    for page in range(1, page_count + 1):
        resp = requests.get(BASE_URL.format(page), timeout=10)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.text, "html.parser")
        for item in soup.find_all("article", class_="product_pod"):
            price_text = item.find("p", class_="price_color").text
            results.append({
                "name"  : item.find("h3").find("a")["title"],
                "price" : float(re.sub(r"[^0-9.]", "", price_text)),
                "rating": RATING_MAP.get(item.find("p", class_="star-rating")["class"][1]),
            })
    return results

def save_to_postgres(data: list[dict], config: dict) -> None:
    # 1) Open connection to NeonDB
    conn = psycopg2.connect(**config)
    cur  = conn.cursor()

    # 2) Create table if it doesn't exist
    cur.execute("""
        CREATE TABLE IF NOT EXISTS products (
            id     SERIAL  PRIMARY KEY,
            name   TEXT    NOT NULL,
            price  NUMERIC,
            rating INTEGER
        )
    """)

    # 3) Insert all rows at once (execute_values is more efficient than executemany)
    psycopg2.extras.execute_values(
        cur,
        "INSERT INTO products (name, price, rating) VALUES %s",
        [(row["name"], row["price"], row["rating"]) for row in data],
    )

    conn.commit()
    print(f"[PostgreSQL / NeonDB] {len(data)} rows saved to the products table")

    # 4) Verify: read back the first 5 rows
    cur.execute("SELECT * FROM products LIMIT 5")
    rows = cur.fetchall()
    print("\nSample rows from NeonDB:")
    for row in rows:
        print(row)

    cur.close()
    conn.close()

# ── run ───────────────────────────────────────────────────────────────────────
data = scrape_and_clean(page_count=2)
print(f"Ready to insert {len(data)} rows into PostgreSQL.\n")

try:
    save_to_postgres(data, PG_CONFIG)
except Exception as e:
    print(f"[INFO] Could not connect to NeonDB: {e}")
    print("Make sure PGHOST, PGUSER, and PGPASSWORD are set to the correct values.")
    print("Set them as environment variables or update PG_CONFIG in the cell above.")

Ready to insert 40 rows into PostgreSQL.



[INFO] Could not connect to NeonDB: connection to server at "ep-silent-bar-adim0vxo-pooler.c-2.us-east-1.aws.neon.tech" (44.198.216.75), port 5432 failed: ERROR:  password authentication failed for user 'sample_user'

Make sure PGHOST, PGUSER, and PGPASSWORD are set to the correct values.
Set them as environment variables or update PG_CONFIG in the cell above.
